<a href="https://colab.research.google.com/github/george-hawkins/japanese/blob/master/upscaling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# @title Connect to Google Drive
from google.colab import auth
from google.colab import drive

# Triggers authentication flow.
auth.authenticate_user()

# And mount drive.
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# @title Install Dependencies
!pip install spandrel Pillow torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 320.8/320.8 kB 17.0 MB/s eta 0:00:00


In [3]:
# @title Select Model
model_details = [
    # ("2x_denoise_FDAT_M_unshuffle_30k_fp16", "2x_IllustrationJaNai_V3denoise_FDAT_M_unshuffle_30k_fp16.safetensors"),
    # ("2x_denoise_SPAN_S_30k_fp16",           "2x_IllustrationJaNai_V3denoise_SPAN_S_30k_fp16.safetensors"),
    # ("4x_denoise_DAT2_27k_bf16",             "4x_IllustrationJaNai_V3denoise_DAT2_27k_bf16.safetensors"),
    # ("4x_denoise_FDAT_M_47k_fp16",           "4x_IllustrationJaNai_V3denoise_FDAT_M_47k_fp16.safetensors"),
    # ("4x_denoise_FDAT_XL_32k_bf16",          "4x_IllustrationJaNai_V3denoise_FDAT_XL_32k_bf16.safetensors"),
    ("2x_detail_FDAT_M_unshuffle_40k_fp16",  "2x_IllustrationJaNai_V3detail_FDAT_M_unshuffle_40k_fp16.safetensors"),
    # ("2x_detail_SPAN_S_40k_fp16",            "2x_IllustrationJaNai_V3detail_SPAN_S_40k_fp16.safetensors"),
    # ("4x_detail_DAT2_28k_bf16",              "4x_IllustrationJaNai_V3detail_DAT2_28k_bf16.safetensors"),
    # ("4x_detail_FDAT_M_40k_fp16",            "4x_IllustrationJaNai_V3detail_FDAT_M_40k_fp16.safetensors"),
    # ("4x_detail_FDAT_XL_27k_bf16",           "4x_IllustrationJaNai_V3detail_FDAT_XL_27k_bf16.safetensors"),
    # ("4x_detail_HAT_L_28k_bf16",             "4x_IllustrationJaNai_V3detail_HAT_L_28k_bf16.safetensors")
]

In [4]:
# @title Check CUDA
import torch
if not torch.cuda.is_available():
    raise RuntimeError("CUDA is not available")

In [5]:
# @title Run Upscaling
import torch
import time
import numpy as np
from PIL import Image
from spandrel import ModelLoader
from pathlib import Path
from datetime import datetime

# Setup
device = torch.device("cuda")

root = "/content/drive/MyDrive/colab/upscaling"
input_dir = Path(f"{root}/input")
output_dir = Path(f"{root}/output")

# Load Model
loader = ModelLoader()

for model_name, model_file in model_details:
    print(f"model: {model_file}")
    model_path = f"{root}/models/{model_file}"
    model = loader.load_from_file(model_path).to(device).eval()

    print(f"Processing images from {input_dir}...")

    for img_path in sorted(input_dir.glob("*")):
        if img_path.suffix.lower() in [".jpg", ".jpeg", ".png", ".webp"]:
            t0 = time.perf_counter()

            # Load image
            img = Image.open(img_path).convert("RGB")
            img_tensor = torch.from_numpy(np.array(img)).permute(2, 0, 1).float().divide(255).unsqueeze(0).to(device)

            # Clear setup so we can gauge VRAM usage.
            torch.cuda.empty_cache()
            torch.cuda.reset_peak_memory_stats()

            # Upscale
            with torch.no_grad():
                output_tensor = model(img_tensor)

            # Output VRAM usage.
            peak_gb = torch.cuda.max_memory_reserved() / 1024**3
            print(f"{img_path.name}: peak VRAM {peak_gb:.2f} GiB")
            elapsed = time.perf_counter() - t0
            print(f"{img_path.name}: {elapsed:.2f}s")

            # Save output
            output_img = output_tensor.squeeze(0).permute(1, 2, 0).clamp(0, 1).cpu().numpy()
            output_img = (output_img * 255).astype(np.uint8)
            Image.fromarray(output_img).save(output_dir / f"{img_path.stem}-{model_name}.png")
            current_time = datetime.now().strftime("%H:%M:%S.%f")[:-3]
            print(f"{current_time} Finished: {img_path.name}")

    print(f"Done! Check the {output_dir} folder.")

model: 2x_IllustrationJaNai_V3detail_FDAT_M_unshuffle_40k_fp16.safetensors
Processing images from /content/drive/MyDrive/colab/upscaling/input...
page-009.png: peak VRAM 29.73 GiB
page-009.png: 5.54s
12:57:34.104 Finished: page-009.png
Done! Check the /content/drive/MyDrive/colab/upscaling/output folder.
